### 1. Input

In [23]:
# ! pip install SpeechRecognition pydub
# ! pip install langdetect
# ! pip install openai-whisper
# ! pip install torch
import speech_recognition as sr
import whisper as w
import torch
from langdetect import detect
import re

#### 1.1. S2T

In [25]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Modelo base
model = w.load_model("base", device=device)

def input_speech(audio = None):
    # Opción 1: Se le pasa un archivo de audio .wav
    if audio != None:
        # Utilizamos whisper porque procesa el archivo directamente, detecta el idioma y lo transcribe (con speech_recognition había que especificar el idioma previamente)
        result = model.transcribe(audio)
        # Se devuelve la transcripción del audio y el idioma
        return result["text"].strip(), result["language"]
    
    # Opción 2: Audio en directo
    else:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            # Para reconocer el nivel de ruido en el audio y eliminar esa parte
            r.adjust_for_ambient_noise(source, duration=1)
            # Se escucha el audio
            audio_data = r.listen(source)
            
            # Guardamos el audio temporalmente ya que whisper lo necesita
            with open("audio_temporal.wav", "wb") as a:
                a.write(audio_data.get_wav_data())
                
            result = model.transcribe("audio_temporal.wav")
            # Devolvemos el texto y el idioma
            return result["text"].strip(), result["language"]

#### 1.2. T2T

In [26]:
def input_text():
    # El usuario escribe sobre qué quiere información
    texto = input("¿Sobre qué quieres información?")

    # Detectamos el idioma
    try:
        idioma = detect(texto)
    except:
        # Idioma por defecto
        idioma = "es"
    
    print(f"Idioma del texto: {idioma}")
    # Devolvemos el texto y el idioma
    return texto, idioma

#### 1.3. Cleaning text

In [27]:
def cleaning_text(texto, idioma):
    # strip para eliminar espacios en blanco
    # capitalize para poner la primera letra en mayúscula
    texto_limpio = texto.strip().capitalize()

    # Muletillas a eliminar (español e inglés)
    muletillas = {
        "es": [r"\btipo+\b", r"\beh+\b", r"\besto+\b", r"\bpues+\b", r"\ben\s+plan\b"],
        "en": [r"\bum+\b", r"\beh+\b", r"\blike\b", r"\byou know\b"]
    }
    
    # Eliminamos las muletillas
    if idioma in muletillas:
        for i in muletillas[idioma]:
            texto_limpio = re.sub(i, "", texto_limpio, flags = re.IGNORECASE)

    # Eliminamos los n espacios que se han creado al eliminar las muletillas y lo sustitumos por un espacio
    texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

    return texto_limpio

In [ ]:
def menu_principal():
    print("Guía turística ")
    print("Opcionesa elegir:")
    print("1. Escribir")
    print("2. Hablar/Audio")
    print("3. Hacer foto")
    print("4. Salir")
    
    opcion = input("\nSelecciona una opción (1, 2, 3 o 4): ")
    
    if opcion == "1":
        texto_sucio, idioma = input_text()
        texto_limpio = cleaning_text(texto_sucio, idioma)
    
    elif opcion == "2":
        print("\n¿Quieres (a) hablar o (b) pasar un archivo .wav?")
        op = input("Seleccion una opción (a o b): ").lower()
        if op == "a":
            print("Escuchando...")
            texto_sucio, idioma = input_speech()
        else:
            ruta = input("Introduce la ruta .wav: ")
            texto_sucio, idioma = input_speech(audio = ruta) 
        texto_limpio = cleaning_text(texto_sucio, idioma)
        
    else:
        print("Opción no válida.")
        return None, None